In [1]:
import torch

# A tensor is just a multi-dimensional array
# The shape tells you exactly what it contains

scalar   = torch.tensor(3.14)                    # shape: []        = one number
vector   = torch.tensor([1.0, 2.0, 3.0])         # shape: [3]       = 3 numbers
matrix   = torch.zeros(4, 5)                     # shape: [4, 5]    = 4 rows, 5 cols
image    = torch.zeros(3, 256, 256)              # shape: [3,256,256] = one RGB image
batch    = torch.zeros(32, 3, 256, 256)          # shape: [32,3,256,256] = 32 images

# Always verify shape before doing anything else
print("scalar:", scalar.shape)     # torch.Size([])
print("vector:", vector.shape)     # torch.Size([3])
print("matrix:", matrix.shape)     # torch.Size([4, 5])
print("image:", image.shape)       # torch.Size([3, 256, 256])
print("batch:", batch.shape)       # torch.Size([32, 3, 256, 256])

scalar: torch.Size([])
vector: torch.Size([3])
matrix: torch.Size([4, 5])
image: torch.Size([3, 256, 256])
batch: torch.Size([32, 3, 256, 256])


In [2]:
# dtype = what kind of numbers are stored
float_tensor = torch.tensor([1.0, 2.0])       # float32 by default
int_tensor   = torch.tensor([1, 2])            # int64 by default
print("float dtype:", float_tensor.dtype)      # torch.float32
print("int dtype:", int_tensor.dtype)          # torch.int64

# Models expect float32 images
# Labels must be int64 (called LongTensor)
image  = torch.randn(3, 256, 256)             # float32 — correct for image
label  = torch.tensor(3)                      # int64 — correct for class index
print("image dtype:", image.dtype)
print("label dtype:", label.dtype)

# device = where is this tensor stored (CPU or GPU)
print("device:", image.device)    # cpu
if torch.cuda.is_available():
    image_gpu = image.cuda()
    print("GPU device:", image_gpu.device)

float dtype: torch.float32
int dtype: torch.int64
image dtype: torch.float32
label dtype: torch.int64
device: cpu
GPU device: cuda:0


In [3]:
# Mistake 1 — model expects batch but you give single image
model_input  = torch.randn(3, 256, 256)        # WRONG — no batch dimension
model_input  = torch.randn(1, 3, 256, 256)     # CORRECT — batch of 1
model_input  = model_input.unsqueeze(0)        # OR: add batch dim with unsqueeze

# Mistake 2 — channels last vs channels first
# PIL Image: (width, height, channels) = (256, 256, 3)
# PyTorch:   (channels, height, width) = (3, 256, 256)
# torchvision ToTensor() handles this conversion automatically

# Mistake 3 — not normalising pixel values
raw    = torch.randint(0, 255, (3, 256, 256)).float()   # values 0-255
norm   = raw / 255.0                                     # values 0-1
# torchvision ToTensor() also does this automatically

print("Always use torchvision.transforms.ToTensor() — it fixes both issues")

Always use torchvision.transforms.ToTensor() — it fixes both issues


In [4]:
import torch
import torch.nn as nn

# ----- Fake data to simulate the problem -----
# Imagine we have 100 "images" (just random vectors here)
# and 100 labels (0 to 6 = 7 classes like our lesion types)
X = torch.randn(100, 10)            # 100 samples, 10 features each
y = torch.randint(0, 7, (100,))     # 100 labels, values 0-6

# ----- Simple model (just for understanding) -----
model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 7)                 # 7 outputs = 7 classes
)

# ----- Loss function -----
criterion = nn.CrossEntropyLoss()    # we will replace this with focal loss later

# ----- Optimizer -----
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# ----- Training loop -----
for epoch in range(10):
    # Step 1: Forward pass — model makes predictions
    predictions = model(X)           # shape: [100, 7]
    
    # Step 2: Compute loss — how wrong were the predictions?
    loss = criterion(predictions, y)
    
    # Step 3: Zero gradients from previous step
    optimizer.zero_grad()
    
    # Step 4: Backward pass — compute gradients
    loss.backward()
    
    # Step 5: Update weights
    optimizer.step()
    
    print(f"Epoch {epoch+1:2d} | Loss: {loss.item():.4f}")

Epoch  1 | Loss: 1.9548
Epoch  2 | Loss: 1.9467
Epoch  3 | Loss: 1.9388
Epoch  4 | Loss: 1.9310
Epoch  5 | Loss: 1.9233
Epoch  6 | Loss: 1.9158
Epoch  7 | Loss: 1.9085
Epoch  8 | Loss: 1.9013
Epoch  9 | Loss: 1.8943
Epoch 10 | Loss: 1.8873


In [5]:
# Let's look at ONE step in slow motion

model2 = nn.Linear(3, 2)         # simplest possible model: 3 inputs, 2 outputs
x = torch.tensor([[1.0, 2.0, 3.0]])    # one sample

print("--- BEFORE training step ---")
print("Weights:", model2.weight.data)
print("Bias:", model2.bias.data)

# Forward pass
out = model2(x)
print("\nPrediction:", out)

# Fake label — let's say class 1 is correct
label = torch.tensor([1])
loss = nn.CrossEntropyLoss()(out, label)
print("Loss:", loss.item())

# Backward pass
loss.backward()
print("\nGradients (how much each weight contributed to the error):")
print(model2.weight.grad)

# Optimizer step — nudge weights to reduce loss
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.1)
optimizer2.step()

print("\n--- AFTER training step ---")
print("Weights:", model2.weight.data)
print("(Notice the weights changed slightly)")

--- BEFORE training step ---
Weights: tensor([[-0.1708,  0.2092, -0.3937],
        [-0.3377, -0.3369,  0.0088]])
Bias: tensor([-0.0799, -0.5458])

Prediction: tensor([[-1.0133, -1.5309]], grad_fn=<AddmmBackward0>)
Loss: 0.9850677251815796

Gradients (how much each weight contributed to the error):
tensor([[ 0.6266,  1.2532,  1.8798],
        [-0.6266, -1.2532, -1.8798]])

--- AFTER training step ---
Weights: tensor([[-0.2334,  0.0839, -0.5817],
        [-0.2751, -0.2116,  0.1968]])
(Notice the weights changed slightly)


In [ ]:
import timm

# EfficientNetV2 was trained on ImageNet (1.2 million images, 1000 classes)
# It already knows: edges, textures, shapes, colours
# We just need to teach it the difference between 7 skin lesion types

# Load pretrained — this downloads ~80MB of weights
model = timm.create_model('tf_efficientnetv2_s', pretrained=True)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")    # ~20 million

# The last layer classifies into 1000 ImageNet classes
# We need to change it to 7 classes
print("\nOriginal final layer:", model.classifier)

# Replace with 7-class classifier
import torch.nn as nn
model.classifier = nn.Linear(model.classifier.in_features, 7)
print("New final layer:", model.classifier)

# Verify output shape
x = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    out = model(x)
print("\nOutput shape:", out.shape)    # torch.Size([1, 7])
print("Now outputs 7 class scores instead of 1000")